In [3]:
import pandas as pd
import joblib
import random
import shap
from sklearn.metrics import f1_score

# Load saved elements
test_df = pd.read_csv('test_split.csv').dropna()
tfidf = joblib.load('tfidf_vectorizer.pkl')
lr_model = joblib.load('logistic_regression_model.pkl')

def inject_visual_homoglyphs(text, perturbation_probability=0.20):
    homoglyphs = {
        'a': ['4', '@', 'α'], 'e': ['3', '€'], 'i': ['1', '!', '|'],
        'o': ['0', 'ø'], 's': ['5', '$'], 't': ['7', '+']
    }
    words = str(text).split()
    perturbed_words = []
    for word in words:
        perturbed_word = ""
        for char in word:
            if char.lower() in homoglyphs and random.random() < perturbation_probability:
                perturbed_word += random.choice(homoglyphs[char.lower()])
            else:
                perturbed_word += char
        perturbed_words.append(perturbed_word)
    return " ".join(perturbed_words)

# Ensure matching keys
X_test_clean = test_df['text']
y_test = test_df['label']
X_test_adversarial = X_test_clean.apply(inject_visual_homoglyphs)

X_test_clean_tfidf = tfidf.transform(X_test_clean)
X_test_adv_tfidf = tfidf.transform(X_test_adversarial)

c:\Users\BLACKBOX\.anaconda-desktop\micromamba\envs\cuda\envs\condaenv1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Clean Eval
preds_clean = lr_model.predict(X_test_clean_tfidf)
f1_clean = f1_score(y_test, preds_clean)

# Adversarial Eval
preds_adv = lr_model.predict(X_test_adv_tfidf)
f1_adv = f1_score(y_test, preds_adv)

print("--- Robustness Performance Gap ---")
print(f"Clean Dataset F1 Score:        {f1_clean:.4f}")
print(f"Adversarial Homoglyph F1 Score: {f1_adv:.4f}")
print(f"Performance Drop Percentage:    {((f1_clean - f1_adv) / f1_clean) * 100:.2f}%")

--- Robustness Performance Gap ---
Clean Dataset F1 Score:        0.5516
Adversarial Homoglyph F1 Score: 0.4323
Performance Drop Percentage:    21.62%


In [5]:
print("Initializing SHAP Explainer...")
# Explain predictions on a small slice to observe token shifting
explain_sample = X_test_clean.iloc[:5]
explain_sample_adv = X_test_adversarial.iloc[:5]

# Fit background distribution
explainer = shap.LinearExplainer(lr_model, tfidf.transform(test_df['text'].iloc[:200]))

# Generate feature importance distributions
shap_values_clean = explainer.shap_values(tfidf.transform(explain_sample))
shap_values_adv = explainer.shap_values(tfidf.transform(explain_sample_adv))

print("SHAP values calculated.")
print("Compare 'shap_values_clean' arrays against 'shap_values_adv' to show feature shifting in your manuscript figures.")

Background dataset has 200 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=200 when initializing the masker.


Initializing SHAP Explainer...
SHAP values calculated.
Compare 'shap_values_clean' arrays against 'shap_values_adv' to show feature shifting in your manuscript figures.
